In [2]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/Qwen-Meeting-Bot"
os.makedirs(output_dir, exist_ok=True)

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [4]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = []

    for convo in convos:
        text = tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False
        )
        texts.append(text)

    return { "text" : texts }

dataset = load_dataset("json", data_files="/content/drive/MyDrive/Qwen-Meeting-Bot/train_clean.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,

    packing = True,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 4,
        warmup_steps = 5,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
model_name = "qwen2.5-7b-mixed-meeting"
save_path = f"{output_dir}/{model_name}"

model.save_pretrained_gguf(save_path, tokenizer, quantization_method = "q4_k_m")

In [ ]:
import os  # save gguf requires more ram, than free colab is providing
from google.colab import drive

drive.mount('/content/drive')

model_dir = "/content/drive/MyDrive/Qwen-Meeting-Bot/qwen2.5-7b-mixed-meeting"
final_output = f"{model_dir}-final.gguf"

!rm -rf llama.cpp llama_bin temp.gguf
!pip uninstall -y gguf > /dev/null

!git clone https://github.com/ggerganov/llama.cpp > /dev/null
!pip install -e llama.cpp/gguf-py > /dev/null
!pip install -r llama.cpp/requirements.txt > /dev/null
!pip install huggingface_hub > /dev/null

!wget -q https://github.com/ggerganov/llama.cpp/releases/download/b3565/llama-b3565-bin-ubuntu-x64.zip
!unzip -o -q llama-b3565-bin-ubuntu-x64.zip -d llama_bin
!cp llama_bin/build/bin/llama-quantize .
!chmod +x llama-quantize

!python llama.cpp/convert_hf_to_gguf.py "{model_dir}" --outfile temp.gguf

if os.path.exists("temp.gguf"):
    !./llama-quantize temp.gguf "{final_output}" q4_k_m
    !rm temp.gguf